In [ ]:
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message=".*torch.cuda.amp.*deprecated.*")

os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning"


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

FEATS_CSV = Path("C:\Users\ironh\OneDrive\Documents\PIG_Behavior_Project\behavior_with_feats_rectROI.csv")
IMG_ROOT  = Path("C:\Users\ironh\OneDrive\Documents\PIG_Behavior_Project\images_clean")

print("FEATS_CSV:", FEATS_CSV)
print("IMG_ROOT :", IMG_ROOT)

assert FEATS_CSV.exists()
assert IMG_ROOT.exists()

df = pd.read_csv(FEATS_CSV)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

BEHAVIORS = [
    "drink",
    "eat",
    "fight",
    "social-nose",
    "explore",
    "lying",
    "stand",
    "move",
    "sitting",
    "playwithtoy",
]
behavior2idx = {b:i for i,b in enumerate(BEHAVIORS)}
idx2behavior = {i:b for b,i in behavior2idx.items()}
NUM_CLASSES  = len(BEHAVIORS)

df["behavior"] = df["behavior"].astype(str)
df = df[df["behavior"].isin(BEHAVIORS)].reset_index(drop=True)
print("After behavior filter rows:", len(df))
print(df["behavior"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

df["group_id"] = df["group_id"].astype(str)
df["pig_id"]   = df["pig_id"].astype(str)

seq_rows = []
for (gid, pid), sub in df.groupby(["group_id","pig_id"]):
    beh_counts = sub["behavior"].value_counts()
    maj = beh_counts.idxmax()
    seq_rows.append({
        "group_id": gid,
        "pig_id": pid,
        "behavior": maj,
        "n_frames": len(sub),
    })

seq_df = pd.DataFrame(seq_rows)
print("Total sequences:", len(seq_df))
print("Sequence behavior dist:")
print(seq_df["behavior"].value_counts())

# encode label
seq_df["label_idx"] = seq_df["behavior"].map(behavior2idx)

labels_all = seq_df["label_idx"].values
indices = np.arange(len(seq_df))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=labels_all
)

seq_df_train = seq_df.iloc[train_idx].reset_index(drop=True)
seq_df_val   = seq_df.iloc[val_idx].reset_index(drop=True)

print("Train seq:", len(seq_df_train))
print("Val   seq:", len(seq_df_val))
print("Train dist:")
print(seq_df_train["behavior"].value_counts())
print("Val dist:")
print(seq_df_val["behavior"].value_counts())

seq_df_train["split"] = "train"
seq_df_val["split"]   = "val"
seq_split = pd.concat([seq_df_train, seq_df_val], ignore_index=True)[["group_id","pig_id","split"]]

df = df.merge(seq_split, on=["group_id","pig_id"], how="inner")
print("Frame-level rows after merge+split:", len(df))
print(df["split"].value_counts())


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np

class BehaviorSequenceDataset(Dataset):
    def __init__(self, df, img_root, behavior2idx, split="train", transform=None):
        """
            img_name, group_id, pig_id, order,
            x1,y1,x2,y2,
            cx_n,cy_n,bw_n,bh_n,
            speed_feat, min_dist_other, num_close_other,
            in_feeder,in_drinker,in_toy,
            behavior, split
        """
        self.img_root = str(img_root)
        self.behavior2idx = behavior2idx
        self.transform = transform

        df = df.copy()
        df = df[df["split"] == split].reset_index(drop=True)

        # Gom sequence theo group_id, pig_id
        self.groups = []
        for (gid, pid), sub in df.groupby(["group_id","pig_id"]):
            if "order" in sub.columns:
                sub = sub.sort_values("order")
            else:
                sub = sub.sort_values("img_name")

            beh = sub["behavior"].iloc[0]
            if beh not in behavior2idx:
                continue

            self.groups.append({
                "group_id": gid,
                "pig_id": pid,
                "behavior": beh,
                "rows": sub.reset_index(drop=True),
            })

        print(f"[SEQ DATASET] {split}: {len(self.groups)} sequences")

        self.extra_cols = [
            "cx_n", "cy_n", "bw_n", "bh_n",
            "speed_feat",
            "min_dist_other", "num_close_other",
            "in_feeder", "in_drinker", "in_toy",
        ]
        self.BURST_LEN = 6

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        rows = g["rows"]
        T = len(rows)

        imgs = []
        feats = []
        for i in range(T):
            row = rows.iloc[i]
            img_path = os.path.join(self.img_root, row["img_name"])
            img = Image.open(img_path).convert("RGB")

            # crop bbox
            x1, y1, x2, y2 = row["x1"], row["y1"], row["x2"], row["y2"]
            img = img.crop((x1, y1, x2, y2))

            if self.transform is not None:
                img_t = self.transform(img)
            else:
                arr = np.array(img).astype("float32") / 255.0
                arr = np.transpose(arr, (2, 0, 1))
                img_t = torch.from_numpy(arr)

            feat = torch.tensor(row[self.extra_cols].values.astype("float32"))
            imgs.append(img_t)
            feats.append(feat)

        while len(imgs) < self.BURST_LEN:
            imgs.append(imgs[-1].clone())
            feats.append(feats[-1].clone())
        imgs = imgs[:self.BURST_LEN]
        feats = feats[:self.BURST_LEN]
        imgs = torch.stack(imgs, dim=0)    # [T,C,H,W]
        feats = torch.stack(feats, dim=0)  # [T,K]
        label = torch.tensor(self.behavior2idx[g["behavior"]], dtype=torch.long)
        return imgs, feats, label


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=spatial_kernel,
                                      padding=spatial_kernel // 2, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        # channel attn
        avg_pool = F.adaptive_avg_pool2d(x, 1).view(B, C)
        max_pool = F.adaptive_max_pool2d(x, 1).view(B, C)
        ca = self.mlp(avg_pool) + self.mlp(max_pool)
        ca = torch.sigmoid(ca).view(B, C, 1, 1)
        x = x * ca
        # spatial attn
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.cat([avg_map, max_map], dim=1)
        sa = torch.sigmoid(self.conv_spatial(sa))
        x = x * sa
        return x

class BehaviorTransformerNet(nn.Module):
    def __init__(
        self,
        num_behaviors,
        extra_dim,
        d_model=320,
        nhead=4,
        num_layers=2,
        backbone_name="resnet34",
        freeze_backbone=True,
        dropout=0.2,
    ):
        super().__init__()

        if backbone_name == "resnet18":
            backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        elif backbone_name == "resnet34":
            backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        else:
            raise ValueError("Unsupported backbone")

        self.cnn = nn.Sequential(*list(backbone.children())[:-2])
        cnn_out_channels = backbone.fc.in_features  # 512 cho resnet18/34

        if freeze_backbone:
            for p in self.cnn.parameters():
                p.requires_grad = False

        self.cbam = CBAMBlock(cnn_out_channels, reduction=16, spatial_kernel=7)

        base_dim = d_model - extra_dim
        if base_dim <= 0:
            raise ValueError()
        self.cnn_proj = nn.Linear(cnn_out_channels, base_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.attn_fc = nn.Linear(d_model, 1)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_behaviors),
        )

    def unfreeze_backbone(self):
        for p in self.cnn.parameters():
            p.requires_grad = True

    def forward(self, seq_imgs, seq_feats):
        B, T, C, H, W = seq_imgs.shape
        x = seq_imgs.view(B*T, C, H, W)

        feat_map = self.cnn(x)
        feat_map = self.cbam(feat_map)
        feat_vec = F.adaptive_avg_pool2d(feat_map, 1).view(B*T, -1)

        feat_vec = self.cnn_proj(feat_vec)
        feat_vec = feat_vec.view(B, T, -1)

        if seq_feats is not None:
            seq_feats = seq_feats.to(feat_vec.dtype)
            h = torch.cat([feat_vec, seq_feats], dim=-1)
        else:
            h = feat_vec

        h_enc = self.transformer(h)
        attn_score = self.attn_fc(h_enc).squeeze(-1)
        attn_weight = torch.softmax(attn_score, dim=1).unsqueeze(-1)
        z = torch.sum(h_enc * attn_weight, dim=1)
        logits = self.head(z)
        return logits


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
from collections import Counter
import numpy as np

train_tf = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.2),
    T.RandomAffine(
        degrees=5,
        translate=(0.05, 0.05),
        scale=(0.9, 1.1),
    ),
    T.ToTensor(),
])

val_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# dataset
train_ds = BehaviorSequenceDataset(df, IMG_ROOT, behavior2idx, split="train", transform=train_tf)
val_ds   = BehaviorSequenceDataset(df, IMG_ROOT, behavior2idx, split="val",   transform=val_tf)

extra_dim = len(train_ds.extra_cols)
print("extra_dim:", extra_dim, "num_classes:", NUM_CLASSES)

train_behaviors_seq = [g["behavior"] for g in train_ds.groups]  # 1 label / sequence

train_counts = (
    pd.Series(Counter(train_behaviors_seq))
    .reindex(BEHAVIORS, fill_value=0)
)
freq = train_counts / train_counts.sum()

weights_cls = 1.0 / np.sqrt(freq.values + 1e-8)
class_weights = torch.tensor(weights_cls, dtype=torch.float32)

print("Train counts (sequence-level):", train_counts.to_dict())
print("Class weights:", class_weights.numpy())

# sample theo sequence
seq_labels_idx = np.array([behavior2idx[b] for b in train_behaviors_seq])
seq_weights = weights_cls[seq_labels_idx]

sampler = WeightedRandomSampler(
    weights=seq_weights.astype("float32"),
    num_samples=len(seq_weights),
    replacement=True,
)

use_sampler = False  # oversample false

BATCH_SIZE = 16

if use_sampler:
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=0,     
        pin_memory=True,
    )
else:
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def visualize_block1_input(dataset, idx=None, title_suffix="train"):
    if idx is None:
        idx = np.random.randint(len(dataset))

    imgs, feats, label = dataset[idx]   # imgs: [T,C,H,W], feats: [T,K]
    T, C, H, W = imgs.shape
    extra_dim = feats.shape[-1]

    imgs_np = imgs.cpu().numpy()
    imgs_np = np.transpose(imgs_np, (0, 2, 3, 1))  # [T,H,W,C]

    fig, axes = plt.subplots(1, T, figsize=(14, 3))
    if T == 1:
        axes = [axes]
    plt.subplots_adjust(bottom=0.25, top=0.8)

    for i in range(T):
        ax = axes[i]
        img = imgs_np[i]
        ax.imshow(img.clip(0, 1))
        ax.set_title(f"k{i}", fontsize=10)
        ax.axis("off")

    plt.show()
    return idx

sample_idx = visualize_block1_input(train_ds)
print("Sample idx:", sample_idx)


In [ ]:
import torch
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm
from pathlib import Path
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class_weights = class_weights.to(device)

model = BehaviorTransformerNet(
    num_behaviors=NUM_CLASSES,
    extra_dim=extra_dim,
    d_model=320,
    nhead=4,
    num_layers=2,
    backbone_name="resnet34",
    freeze_backbone=True,
    dropout=0.2,
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

USE_AMP = True
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

NUM_EPOCHS    = 50
PATIENCE      = 8
WARMUP_EPOCHS = 5
max_grad_norm = 5.0

CHECKPOINT_PATH = Path("/kaggle/working/pig_behavior_best_model.pt")

best_macroF1 = 0.0
best_state   = None
no_improve   = 0
backbone_unfrozen = False
start_epoch = 1

if CHECKPOINT_PATH.exists():
    print(f"Found checkpoint at {CHECKPOINT_PATH}, loading...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)

    if isinstance(ckpt, dict) and "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optim_state"])
        scaler.load_state_dict(ckpt["scaler_state"])

        best_macroF1 = ckpt.get("best_macroF1", 0.0)
        no_improve = ckpt.get("no_improve", 0)
        backbone_unfrozen = ckpt.get("backbone_unfrozen", False)
        last_epoch = ckpt.get("epoch", 0)
        start_epoch = last_epoch + 1

        print(f">> Resume from epoch {last_epoch}, best_macroF1={best_macroF1:.3f}, "
              f"no_improve={no_improve}, backbone_unfrozen={backbone_unfrozen}")
    else:
        print(">> Old-format checkpoint detected (only model weights). "
              "Loading model, but optimizer/scaler will start fresh.")
        model.load_state_dict(ckpt)
        best_macroF1 = 0.0
        no_improve = 0
        backbone_unfrozen = False
        start_epoch = 1
else:
    print("No checkpoint found, training from scratch.")

def save_checkpoint(epoch, model, optimizer, scaler,
                    best_macroF1, no_improve, backbone_unfrozen, path):
    ckpt = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_macroF1": best_macroF1,
        "no_improve": no_improve,
        "backbone_unfrozen": backbone_unfrozen,
    }
    torch.save(ckpt, path)
    print(f"  -> Saved checkpoint to {path}")

# train
for epoch in range(start_epoch, NUM_EPOCHS+1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    # Unfreeze backbone sau warmup
    if (not backbone_unfrozen) and (epoch > WARMUP_EPOCHS):
        model.unfreeze_backbone()
        for g in optimizer.param_groups:
            g["lr"] = 5e-5
        backbone_unfrozen = True
        print(">> Unfreeze backbone & lower LR to 5e-5")

    # train
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for seq_imgs, seq_feats, labels in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        seq_imgs = seq_imgs.to(device)
        seq_feats = seq_feats.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(seq_imgs, seq_feats)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= max(train_total, 1)
    train_acc  = train_correct / max(train_total, 1)

    # val
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    all_labels = []
    all_preds  = []

    with torch.inference_mode():
        for seq_imgs, seq_feats, labels in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
            seq_imgs = seq_imgs.to(device)
            seq_feats = seq_feats.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(seq_imgs, seq_feats)
                loss = criterion(logits, labels)

            val_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    val_loss /= max(val_total, 1)
    val_acc  = val_correct / max(val_total, 1)
    macroF1  = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch} | train_loss={train_loss:.4f} acc={train_acc:.3f} "
          f"| val_loss={val_loss:.4f} acc={val_acc:.3f} macroF1={macroF1:.3f}")

    # Early stopping theo macroF1
    if macroF1 > best_macroF1 + 1e-4:
        best_macroF1 = macroF1
        no_improve = 0
        print(f"  -> New best macroF1={best_macroF1:.3f}")
        save_checkpoint(epoch, model, optimizer, scaler,
                        best_macroF1, no_improve, backbone_unfrozen,
                        CHECKPOINT_PATH)
    else:
        no_improve += 1
        print(f"  -> No improvement for {no_improve} epoch(s) (best macroF1={best_macroF1:.3f})")

    if no_improve >= PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs (patience={PATIENCE}).")
        break

print("\nTraining finished.")
print("Best val macro-F1:", best_macroF1)

if CHECKPOINT_PATH.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    if isinstance(ckpt, dict) and "model_state" in ckpt:
        model.load_state_dict(ckpt["model_state"])
    else:
        model.load_state_dict(ckpt)
    print("Loaded best model weights from checkpoint for evaluation/inference.")


In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_labels = []
all_preds  = []

with torch.inference_mode():
    for seq_imgs, seq_feats, labels in tqdm(val_loader, desc="Final eval"):
        seq_imgs = seq_imgs.to(device)
        seq_feats = seq_feats.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(seq_imgs, seq_feats)
        preds = logits.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

print("Final validation classification report (fine behaviors):")
print(classification_report(
    all_labels,
    all_preds,
    target_names=BEHAVIORS,
    digits=3,
))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm
import os

# load best checkpoint
if "CHECKPOINT_PATH" in globals() and os.path.exists(CHECKPOINT_PATH):
    print(f"[LOAD] best checkpoint from {CHECKPOINT_PATH}")
    state = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(state)
else:
    pass

model.eval()

all_labels = []
all_preds  = []

with torch.inference_mode():
    for seq_imgs, seq_feats, labels in tqdm(val_loader, desc="Final eval", leave=True):
        seq_imgs = seq_imgs.to(device)
        seq_feats = seq_feats.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(seq_imgs, seq_feats)

        preds = logits.argmax(dim=1)

        all_labels.extend(labels.cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)

acc      = (all_labels == all_preds).mean()
macroF1  = f1_score(all_labels, all_preds, average="macro")
weightedF1 = f1_score(all_labels, all_preds, average="weighted")

print("\nFINAL METRICS (val set)")
print(f"Accuracy    : {acc:.3f}")
print(f"Macro F1    : {macroF1:.3f}")
print(f"Weighted F1 : {weightedF1:.3f}")

print("\nClassification report (per behavior)")
print(classification_report(all_labels, all_preds, target_names=BEHAVIORS, digits=3))

# ====== 3. Confusion matrix (raw + normalized) ======
cm = confusion_matrix(all_labels, all_preds, labels=np.arange(len(BEHAVIORS)))
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True).clip(min=1.0)

def plot_confusion_matrix(cm, classes, normalize=False, title="Confusion matrix"):
    if normalize:
        cm_display = cm_norm
    else:
        cm_display = cm

    plt.figure(figsize=(8, 7))
    im = plt.imshow(cm_display, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha="right")
    plt.yticks(tick_marks, classes)

    fmt = ".2f" if normalize else "d"
    thresh = cm_display.max() / 2.0
    for i in range(cm_display.shape[0]):
        for j in range(cm_display.shape[1]):
            plt.text(
                j, i,
                format(cm_display[i, j], fmt),
                horizontalalignment="center",
                color="white" if cm_display[i, j] > thresh else "black",
                fontsize=8
            )

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

# Raw confusion matrix
plot_confusion_matrix(cm, BEHAVIORS, normalize=False, title="Confusion Matrix (Counts)")

plot_confusion_matrix(cm, BEHAVIORS, normalize=True, title="Confusion Matrix (Row-normalized)")

report_dict = classification_report(
    all_labels, all_preds,
    target_names=BEHAVIORS,
    output_dict=True
)

per_class_f1 = [report_dict[b]["f1-score"] for b in BEHAVIORS]
per_class_support = [report_dict[b]["support"] for b in BEHAVIORS]

plt.figure(figsize=(9, 5))
x = np.arange(len(BEHAVIORS))
plt.bar(x, per_class_f1)
plt.xticks(x, BEHAVIORS, rotation=45, ha="right")
plt.ylim(0, 1.0)
plt.ylabel("F1-score")
plt.title("F1-score theo tng hnh vi (val set)")
for i, v in enumerate(per_class_f1):
    plt.text(i, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(x, per_class_support)
plt.xticks(x, BEHAVIORS, rotation=45, ha="right")
plt.ylabel("S sequence (support)")
plt.title("S sequence per behavior trong val set")
plt.tight_layout()
plt.show()
